In [ ]:
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal

In [ ]:
load_dotenv()

In [ ]:
llama = ChatOllama(model='llama3.1:8b',temperature=0.5)
gemini = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite',temperature=0.5)

In [ ]:
class Sentiment(BaseModel):
    sentiment : Literal['Positive','Negative'] = Field(description='Provide sentiment for the provided feedback as Positive or Negative')

In [ ]:
pydanticparser = PydanticOutputParser(pydantic_object=Sentiment)
str_parser = StrOutputParser()

In [ ]:
main_prompt = PromptTemplate(template='Generate a Sentiment over provided feedback {feedback} from customer in certain {format} ',input_variables=['feedback'], partial_variables = {'format':pydanticparser.get_format_instructions()})

In [34]:
sentiment_chain = main_prompt | gemini | pydanticparser

In [43]:
positive_prompt = PromptTemplate(template='Generate just one positive response for the provided feedback {feedback} to the customer',input_variables=['feedback'])

In [44]:
negative_prompt = PromptTemplate(template='Generate just one sorry note for the provided {feedback} to the customer' ,
                                 input_variables=['feedback'])

In [45]:
chain_with_sentiment = RunnablePassthrough.assign(sentiment_data = sentiment_chain)

In [46]:
conditional_chain = RunnableBranch((lambda x:x['sentiment_data'].sentiment == 'Positive', positive_prompt | llama | str_parser),(lambda x:x['sentiment_data'].sentiment == 'Negative', negative_prompt | llama | str_parser),RunnableLambda(lambda x:'No sentiments catched from the feedback'))

In [47]:
final_chain = chain_with_sentiment | conditional_chain | str_parser

In [54]:
response = final_chain.invoke({'feedback':'I do not like the phone at all, Camera is Waste'})

In [55]:
response

'Here\'s a possible sorry note:\n\n"Dear Customer,\n\nI am truly sorry to hear that our phone and camera did not meet your expectations. We understand that this has caused frustration and disappointment.\n\nPlease accept my sincerest apologies for any inconvenience this may have caused. If you would like, we can discuss options for a replacement or refund.\n\nThank you for bringing this to our attention, and I hope you will give us the opportunity to serve you better in the future.\n\nSincerely,\n[Your Name]"'